In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)


Mounted at /content/gdrive


In [2]:
import os
path = '/content/gdrive/MyDrive/Trading_Portfolios/'
os.makedirs(path, exist_ok=True) # Ensure the directory exists
print(os.listdir(path))

['actual trades.csv', 'personal trades.csv']


In [1]:
from google.colab import drive
import os
import yfinance as yf
import pandas as pd

portfolio = pd.DataFrame(columns=[
    "Symbol", "Shares", "Buy_Price", "Date_Bought",
    "Sell_Price", "Date_Sold", "Profit_Loss", "Status",
    "Current_Price", "Current_Profit_Loss"
])


SAVE_FOLDER = "/content/drive/MyDrive/Trading_Portfolios/"


# Define FILE_PATH for initial load
FILE_PATH = os.path.join(SAVE_FOLDER, "default_portfolio.csv")

current_file = None


# ---------------- LOAD AT START ----------------
#please dont crash this took forever
expected_columns = [
    "Symbol", "Shares", "Buy_Price", "Date_Bought",
    "Sell_Price", "Date_Sold", "Profit_Loss", "Status",
    "Current_Price", "Current_Profit_Loss"
]

if os.path.exists(FILE_PATH):
    portfolio = pd.read_csv(FILE_PATH)
    # Ensure all expected columns are present after loading
    for col in expected_columns:
        if col not in portfolio.columns:
            portfolio[col] = None
    current_file = FILE_PATH # Set current_file if loaded from FILE_PATH
    print("Existing portfolio loaded.")
else:
    portfolio = pd.DataFrame(columns=expected_columns)
    print("New portfolio created.")

# ---------------- MAIN LOOP i have no idea what this means but sounds cool so ----------------
while True:

    action = input("\nChoose: BUY, SELL, EDIT, VIEW, ANALYZE, SAVE, LOAD, NEW, EXIT: ").upper()

    # ---------------- BUY ----------------
    if action == "BUY":
        symbol = input("Stock symbol (Make sure this is typed correctly!): ").upper()
        shares = float(input("Number of shares: "))
        buy_price_str = input("Buy price: ")
        buy_price = float(buy_price_str.replace('$', '').replace(',', '')) # Clean input
        date_bought = input("Date bought (MM/DD/YYYY): ")

        new_trade = pd.DataFrame([[symbol, shares, buy_price, date_bought,
                                   None, None, None, "OPEN", None, None]],
                                 columns=portfolio.columns)


        portfolio = pd.concat([portfolio, new_trade], ignore_index=True)
        print("Buy order logged successfully! Dont forget to SAVE your work before exiting:) (Status: OPEN).")

    # ---------------- SELL ----------------
    elif action == "SELL":

        open_trades = portfolio[portfolio["Status"] == "OPEN"]

        if open_trades.empty:
            print("No open trades available.")
        else:
            print("\nOpen Trades:")
            print(open_trades)

            index = int(input("Enter index of trade to sell(number on the leftmost column): "))

            if index in portfolio.index and portfolio.loc[index, "Status"] == "OPEN":

                sell_price_str = input("Enter sell price: ")
                sell_price = float(sell_price_str.replace('$', '').replace(',', '')) # Clean input
                date_sold = input("Enter date when shares sold (MM/DD/YYYY): ")

                buy_price = portfolio.loc[index, "Buy_Price"]
                shares = portfolio.loc[index, "Shares"]

                profit_loss = (sell_price - buy_price) * shares

                portfolio.loc[index, "Sell_Price"] = sell_price
                portfolio.loc[index, "Date_Sold"] = date_sold
                portfolio.loc[index, "Profit_Loss"] = profit_loss
                portfolio.loc[index, "Status"] = "CLOSED"

                print("Trade closed successfully. Profit/Loss will be calculated automatically for you :) Please don't forget to press SAVE!")
                print("Profit/Loss:", round(profit_loss, 2))

            else:
                print("Invalid index selected.")

    # ---------------- Analyzing thingy----------------
    elif action == "ANALYZE":
        if portfolio.empty:
            print("No trades to analyze.")
        else:
            # Update open trades first
            open_trades = portfolio[portfolio["Status"] == "OPEN"]
            for idx in open_trades.index:
                symbol = portfolio.loc[idx, "Symbol"]
                try:
                    data = yf.Ticker(symbol)
                    current_price = data.history(period="1d")["Close"].iloc[-1]
                    portfolio.loc[idx, "Current_Price"] = current_price
                    portfolio.loc[idx, "Current_Profit_Loss"] = (
                        current_price - portfolio.loc[idx, "Buy_Price"]
                    ) * portfolio.loc[idx, "Shares"]
                except Exception as e:
                    portfolio.loc[idx, "Current_Price"] = None
                    portfolio.loc[idx, "Current_Profit_Loss"] = None

            # ----- ANALYSIS ONLY FOR CLOSED TRADES -----
            closed_trades = portfolio[portfolio["Status"] == "CLOSED"]

            if closed_trades.empty:
                print("No closed trades to analyze.")
            else:
                total_profit = closed_trades["Profit_Loss"].sum()
                win_rate = (closed_trades["Profit_Loss"] > 0).mean() * 100

                avg_win = closed_trades[closed_trades["Profit_Loss"] > 0]["Profit_Loss"].mean()
                avg_loss = closed_trades[closed_trades["Profit_Loss"] < 0]["Profit_Loss"].mean()

                # User-friendly display for NaN
                avg_win_display = "No winning trades yet" if pd.isna(avg_win) else round(avg_win, 2)
                avg_loss_display = "No losing trades yet" if pd.isna(avg_loss) else round(avg_loss, 2)

                best_trade = closed_trades["Profit_Loss"].max()
                worst_trade = closed_trades["Profit_Loss"].min()

                # Most profitable stock
                symbol_profit = closed_trades.groupby("Symbol")["Profit_Loss"].sum()
                best_symbol = symbol_profit.idxmax() if not symbol_profit.empty else "N/A"

                print("\n----- AI TRADE ANALYSIS -----")
                print("Total Profit/Loss:", round(total_profit, 2))
                print("Win Rate: {:.2f}%".format(win_rate))
                print("Average Win:", avg_win_display)
                print("Average Loss:", avg_loss_display)
                print("Best Trade:", round(best_trade, 2))
                print("Worst Trade:", round(worst_trade, 2))
                print("Most profitable stock:", best_symbol)


    # ---------------- EDIT ----------------
    elif action == "EDIT":
        if portfolio.empty:
            print("Portfolio is currently empty. Make a trade! ")
        else:
            print("\nCurrent Portfolio:")
            print(portfolio)

            index = int(input("Enter index to edit (this is the number on the leftmost column): "))

            if index in portfolio.index:
                column = input("Enter column (name of column you would like to modify, please make sure you type it as is) to edit: ")

                if column in portfolio.columns:
                    new_value = input("Enter new value: ")

                    if column in ["Shares", "Buy_Price", "Sell_Price", "Profit_Loss"]:
                        new_value_cleaned = new_value.replace('$', '').replace(',', '')
                        new_value = float(new_value_cleaned)

                    portfolio.loc[index, column] = new_value

                    if portfolio.loc[index, "Status"] == "CLOSED":
                        buy_price = portfolio.loc[index, "Buy_Price"]
                        sell_price = portfolio.loc[index, "Sell_Price"]
                        shares = portfolio.loc[index, "Shares"]
                        portfolio.loc[index, "Profit_Loss"] = (sell_price - buy_price) * shares

                    print("Trade updated successfully :)")
                else:
                    print("Invalid column name. Please try again!")
            else:
                print("Invalid index. Please try again!")

    # ---------------- VIEW ----------------
    elif action == "VIEW":
        if portfolio.empty:
            print("Portfolio is empty. Make a trade!")
        else:
            # ----- UPDATE OPEN TRADES BEFORE VIEWING -----
            open_trades = portfolio[portfolio["Status"] == "OPEN"]
            for idx in open_trades.index:
                symbol = portfolio.loc[idx, "Symbol"]
                try:
                    data = yf.Ticker(symbol)
                    current_price = data.history(period="1d")["Close"].iloc[-1]
                    portfolio.loc[idx, "Current_Price"] = current_price
                    portfolio.loc[idx, "Current_Profit_Loss"] = (
                        current_price - portfolio.loc[idx, "Buy_Price"]
                    ) * portfolio.loc[idx, "Shares"]
                except Exception as e:
                    portfolio.loc[idx, "Current_Price"] = None
                    portfolio.loc[idx, "Current_Profit_Loss"] = None

            print("\nFull Portfolio:")
            print(portfolio)

    # ---------------- SAVE ----------------
    elif action == "SAVE":
        if current_file is None:
            file_name = input("Enter portfolio name: ")
            current_file = os.path.join(SAVE_FOLDER, file_name + ".csv")

        portfolio.to_csv(current_file, index=False)
        print(f"Portfolio saved to {current_file}")


    # ---------------- LOAD ----------------
    elif action == "LOAD":

        file_name = input("Enter portfolio name (without .csv, for ex: actual trades): ")
        file_path = SAVE_FOLDER + file_name + ".csv"

        if os.path.exists(file_path):
            portfolio = pd.read_csv(file_path)
            # this is bc this issue always shows up💀
            for col in expected_columns:
                if col not in portfolio.columns:
                    portfolio[col] = None
            current_file = file_path
            print("Portfolio loaded successfully.")
        else:
            print("File not found.")



    # ---------------- NEW ----------------

    elif action == "NEW":

        file_name = input("Enter name for new portfolio (please make sure you save this!): ")

        portfolio = pd.DataFrame(columns=expected_columns)

        current_file = SAVE_FOLDER + file_name + ".csv"

        portfolio.to_csv(current_file, index=False) # Automatically save the new, empty portfolio
        print(f"New portfolio '{current_file}' created and saved.")


    # ---------------- EXIT ----------------
    elif action == "EXIT":
        print("""Exiting program. Thank you for using this tracker :) We hope to see you again!\n-investment_tracker 2026 designed by Anna Song. All rights reserved. No part of this code or service may be reproduced for commercial or personal uses.""")
        break

    else:
        print("Invalid option.")

New portfolio created.

Choose: BUY, SELL, EDIT, VIEW, ANALYZE, SAVE, LOAD, NEW, EXIT: view
Portfolio is empty. Make a trade!

Choose: BUY, SELL, EDIT, VIEW, ANALYZE, SAVE, LOAD, NEW, EXIT: load
Enter portfolio name (without .csv, for ex: actual trades): personal trades
Portfolio loaded successfully.

Choose: BUY, SELL, EDIT, VIEW, ANALYZE, SAVE, LOAD, NEW, EXIT: view

Full Portfolio:
   Symbol  Shares  Buy_Price Date_Bought  Sell_Price  Date_Sold  Profit_Loss  \
0     SPY    21.0     687.90   1/12/2026         NaN        NaN          NaN   
1    SIDU    10.0       3.50   1/12/2026         NaN        NaN          NaN   
2    UAMY    20.0       8.61   1/12/2026         NaN        NaN          NaN   
3     AMD    10.0     203.30   1/09/2026      217.98  1/13/2026        146.8   
4     BAC    10.0      54.80   1/12/2026       54.85  1/13/2026          0.5   
5     JPM    10.0     329.25   1/09/2026      310.90  1/14/2026       -183.5   
6     BLK     1.0    1163.17   1/17/2026         NaN

In [5]:
import os

SAVE_FOLDER = "/content/drive/MyDrive/Trading_Portfolios/"

# Ensure the folder exists (it should, as per previous runs)
os.makedirs(SAVE_FOLDER, exist_ok=True)

print(f"Files in '{SAVE_FOLDER}':")
if os.listdir(SAVE_FOLDER):
    for item in os.listdir(SAVE_FOLDER):
        print(item)
else:
    print("No files found in this folder.")

Files in '/content/drive/MyDrive/Trading_Portfolios/':
actual trades.csv
personal trades.csv


Once you run the cell above, you will see a list of your saved portfolios. When the main program prompts you to `Enter portfolio name (without .csv, for ex: actual trades):`, you should enter the exact name of the file you want to load, *without* the `.csv` extension.
